In [1]:
import os

stream_path = "/home/jovyan/dataset/stream_input"
checkpoint_path = "/home/jovyan/work/checkpoints"

# Создаем папки, если их нет
os.makedirs(stream_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

print(f"✅ Папка для потока готова: {stream_path}")
print(f"✅ Папка для чекпоинтов готова: {checkpoint_path}")

✅ Папка для потока готова: /home/jovyan/dataset/stream_input
✅ Папка для чекпоинтов готова: /home/jovyan/work/checkpoints


In [ ]:
import os, time, torch, cv2, numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
import segmentation_models_pytorch as smp

# --- 1. ГЛОБАЛЬНЫЕ НАСТРОЙКИ ---
MODEL_PATH = "/home/jovyan/work/models/deeplabv3plus_best.pth"
SOURCE_IMG_DIR = "/home/jovyan/dataset/Detection/images/val/"
LIVE_STREAM_PATH = "/home/jovyan/work/output_viz/live_stream.jpg"
CHECKPOINT_PATH = "/home/jovyan/work/checkpoint_performance_test"

# Схема входящих сообщений из Kafka
kafka_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("stream_id", StringType(), True), # ID потока для теста
    StructField("filename", StringType(), True),
    StructField("timestamp", DoubleType(), True)
])

# --- 2. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ (DEEPLABV3+) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_model(path):
    model = smp.DeepLabV3Plus(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=3)
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device).eval()
    return model

my_model = load_model(MODEL_PATH)

preprocess = transforms.Compose([
    transforms.Resize((640, 480)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --- 3. ЛОГИКА СЕГМЕНТАЦИИ ---
def run_segmentation(filename):
    full_path = os.path.join(SOURCE_IMG_DIR, filename)
    if not os.path.exists(full_path): return False
    
    try:
        img_pil = Image.open(full_path).convert("RGB")
        input_tensor = preprocess(img_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            output = my_model(input_tensor)
            mask = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype('uint8')

        # Визуализация для LIVE-потока
        frame_cv = cv2.resize(cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR), (480, 640))
        overlay = np.zeros_like(frame_cv)
        # Цвета: Вода (Зеленый), Небо (Синий), Объекты (Желтый)
        overlay[mask == 0], overlay[mask == 1], overlay[mask == 2] = [0,255,0], [255,0,0], [0,255,255]
        annotated = cv2.addWeighted(frame_cv, 0.6, overlay, 0.4, 0)
        
        # Атомарное обновление файла трансляции
        temp_live = LIVE_STREAM_PATH + ".tmp"
        cv2.imwrite(temp_live, annotated)
        os.replace(temp_live, LIVE_STREAM_PATH)
        return True
    except:
        return False

# --- 4. ОБРАБОТКА БАТЧА И ЗАМЕР СКОРОСТИ ---
def process_batch(batch_df, batch_id):
    if batch_df.isEmpty(): return
    
    # --- СТАРТ ЗАМЕРА ---
    batch_start_time = time.time()
    
    pdf = batch_df.toPandas()
    num_frames = len(pdf)
    
    for index, row in pdf.iterrows():
        run_segmentation(row['filename'])
    
    # --- КОНЕЦ ЗАМЕРА ---
    batch_end_time = time.time()
    duration = batch_end_time - batch_start_time
    fps_real = num_frames / duration
    
    # ВЫВОД МЕТРИКИ (Лови эти строки в консоли!)
    print(f"\n" + "="*50)
    print(f"📊 ОТЧЕТ ПО БАТЧУ №{batch_id}")
    print(f"Обработано кадров: {num_frames}")
    print(f"Общее время батча: {duration:.3f} сек")
    print(f"Производительность: {fps_real:.2f} кадра/сек (FPS)")
    print(f"Задержка на 1 кадр: {duration/num_frames:.4f} сек")
    print("="*50 + "\n")

# --- 5. ЗАПУСК SPARK STREAMING ---
spark = SparkSession.builder \
    .appName("Marine_Performance_Test") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# Чтение из Kafka
df_kafka = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "marine_stream") \
    .option("startingOffsets", "latest") \
    .load()

# Парсинг JSON
processed_df = df_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), kafka_schema).alias("data")).select("data.*")

# Запуск потока с выводом в наш профайлер
query = processed_df.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime='1 second') \
    .start()

print("🚢 Spark бенчмарк запущен. Ожидание данных...")
query.awaitTermination()

🚢 Spark бенчмарк запущен. Ожидание данных...

📊 ОТЧЕТ ПО БАТЧУ №1
Обработано кадров: 3
Общее время батча: 7.033 сек
Производительность: 0.43 кадра/сек (FPS)
Задержка на 1 кадр: 2.3443 сек


📊 ОТЧЕТ ПО БАТЧУ №2
Обработано кадров: 83
Общее время батча: 62.802 сек
Производительность: 1.32 кадра/сек (FPS)
Задержка на 1 кадр: 0.7567 сек


📊 ОТЧЕТ ПО БАТЧУ №3
Обработано кадров: 382
Общее время батча: 280.547 сек
Производительность: 1.36 кадра/сек (FPS)
Задержка на 1 кадр: 0.7344 сек


📊 ОТЧЕТ ПО БАТЧУ №4
Обработано кадров: 1629
Общее время батча: 1190.407 сек
Производительность: 1.37 кадра/сек (FPS)
Задержка на 1 кадр: 0.7308 сек

